In [141]:
from datasets import load_dataset
import pandas as pd
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.utils.data import Dataset, DataLoader
from torchinfo import summary
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModel
from torch.nn.utils.rnn import pad_sequence

device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

### How the data will flow in the project 

Load dataset

        ↓
Explore the data

        ↓

Train/Validation/Test split

        ↓

Build vocabulary (using ONLY the training set)

        ↓

Tokenize the text

        ↓

Convert tokens to integer IDs

        ↓

Pad/Truncate sequences

        ↓

Convert to PyTorch tensors

        ↓

Create a custom Dataset

        ↓

Create a DataLoader

        ↓

Build the LSTM model

        ↓

Train

        ↓

Evaluate

In [142]:
data = load_dataset("fancyzhx/ag_news")

In [143]:
# load_dataset returns a datasetDict which containes multiple splits such as: train and test, and it does not have
# .to_pandas() methods. So, you have to convert each split alone. 
print(data)
train_dataset = data["train"]
test_dataset = data["test"]
print(train_dataset[0])
print(test_dataset[0])

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})
{'text': "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.", 'label': 2}
{'text': "Fears for T N pension after talks Unions representing workers at Turner   Newall say they are 'disappointed' after talks with stricken parent firm Federal Mogul.", 'label': 2}


### Data Preprocessing 

In [144]:
# Convert the train split to pandas
train_df = data['train'].to_pandas()
test_df = data['test'].to_pandas()

In [145]:
print(train_df.shape)
print("===========")
print(test_df.shape)

(120000, 2)
(7600, 2)


In [146]:
print(train_df.head())
print(test_df.head())

                                                text  label
0  Wall St. Bears Claw Back Into the Black (Reute...      2
1  Carlyle Looks Toward Commercial Aerospace (Reu...      2
2  Oil and Economy Cloud Stocks' Outlook (Reuters...      2
3  Iraq Halts Oil Exports from Main Southern Pipe...      2
4  Oil prices soar to all-time record, posing new...      2
                                                text  label
0  Fears for T N pension after talks Unions repre...      2
1  The Race is On: Second Private Team Sets Launc...      3
2  Ky. Company Wins Grant to Study Peptides (AP) ...      3
3  Prediction Unit Helps Forecast Wildfires (AP) ...      3
4  Calif. Aims to Limit Farm-Related Smog (AP) AP...      3


In [147]:
train_df.isnull().sum()

text     0
label    0
dtype: int64

In [148]:
train_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 120000 entries, 0 to 119999
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype
---  ------  --------------   -----
 0   text    120000 non-null  str  
 1   label   120000 non-null  int64
dtypes: int64(1), str(1)
memory usage: 28.9 MB


In [149]:
# Training 
X_train = np.array(train_df.iloc[:, 0]) # Training text, its a list of sentences not only one sentence.
print(X_train[0])
y_train = np.array(train_df.iloc[:, 1]) # Training label 
print(y_train[0])

# Testing 
X_test = np.array(test_df.iloc[:, 0]) # Testing text
print(X_test[0])
y_test = np.array(test_df.iloc[:, 1]) # Testing label
print(y_test[0])

Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\band of ultra-cynics, are seeing green again.
2
Fears for T N pension after talks Unions representing workers at Turner   Newall say they are 'disappointed' after talks with stricken parent firm Federal Mogul.
2


In [150]:
# The data is already splitted into training and testing.
# But we need to split the testing dataset into validation data and testing data.

# Split the test data into testing and validation
X_test, X_val, y_test, y_val = train_test_split(X_test, y_test, test_size=0.5)

#### LSTM  cannot read English, it only understand numbers, so we will have to turn the words into numbers.

#### Step 1. Building the Vocabulary

- Vocabulary: simply a unique dictionary of all the words your model knows.
1. You read all the training sentences once. 
2. You write down every unique word.
3. Assign each word a number.
- Why does vocabulary need to be unique? Because the model needs a mapping from words to numbers.

#### Step 2. Tokenization

- A token is simply one unit of text, usually a token is one word. 
- The process of splitting text into tokens is called **tokenization**.
- There are different types of tokenization, like: word tokenization or character tokenization.
- In **Step 1** we have the words as individuals, and the vocabulary dictionary is words not sentences, so we must first split the sentences into words i.e. tokenization.
- PyTorch can not convert raw text into tensors, it should be converted into nubmers first, then tokenized.

#### Step 3. Convert Tokens into IDs

- Now we combine the first two steps. Replace every word using the vocabulary. Now a sentence becomes like that:
[0,1,2,3]

- So we will use the dictionary (translation table) to replace each word with its ID.


#### What about words we have never seen them before:

##### For example: Tesla.

- It does not exits in the vocabulary. 
- We create a special token called UNK --> Unknown word. 
- We give it a unique number (ID).

#### Step 4. Padding 

- When training, every sentence has a different length. For example: [3,5,8] and [6,10,8,9,2,0,1].
- A batch simply contain multiple training examples the model processes together (sentences in your case). 
- Neural networks process data in batches, this is way much faster for the model.
- PyTorch stores the batch in a single tenosr. 
- A tensor requires every row to have the same length (rectangular tensors). 
- Since sentences naturally have different lengths they will not be rectangular tensors like this:
- [
  
    [3, 5, 8],

    [6, 10, 2, 8, 1, 3, 9]

]

- Padding fixes this:
    1. We choose a maximum length.
    2. Add a special value called PAD until it reaches length 7 for sentences that are less then 7.
    3. Usually PAD = 0.
    4. When both sentences are have the same length, Now PyTorch can make one tensor. 
    5. Now the **DataLoader** can give this whole batch to the LSTM at once.
- If the sentence is too long we cut it down using **Truncation**.

In [151]:
# Step 1: Building the tokeniztion list.

# The corpus is all articles, which is X_train. 

# When creating a vocabulary, we want to have a list of words, not a list of lists. 
# append() adds sentence by sentence (which will create a list (the vocabulary) of lists (sentences of corpus)).
# extend() when we want to merge a another collection into a list. so it will add the sentences without being a list.  
# For NLP its prefered to use extend() to create the vocabulary. 

words = [] # A list containign all tokens from the training set. You can call it tokenized corpus.

for sentence in X_train:
    word = sentence.split() # Tokenization
    words.extend(word)

print(words[:45])
print(len(words))

['Wall', 'St.', 'Bears', 'Claw', 'Back', 'Into', 'the', 'Black', '(Reuters)', 'Reuters', '-', 'Short-sellers,', 'Wall', "Street's", 'dwindling\\band', 'of', 'ultra-cynics,', 'are', 'seeing', 'green', 'again.', 'Carlyle', 'Looks', 'Toward', 'Commercial', 'Aerospace', '(Reuters)', 'Reuters', '-', 'Private', 'investment', 'firm', 'Carlyle', 'Group,\\which', 'has', 'a', 'reputation', 'for', 'making', 'well-timed', 'and', 'occasionally\\controversial', 'plays', 'in', 'the']
4541694


In [152]:
# Step 2: Building the vocabulary

# A vocabulary is the unique set of words your model knows.
# As the tokens list is now have a lot of duplicates, we need to remove them so we can map each word to its ID.
# This is called Vocabulary mapping or Word-to-index dictionary. 

unique_vocabulary = set(words) # unique vocab


In [ ]:
# set --> list

# Now we need to convert the set into a list, because a set has not fiex order.
# We need a fixed order. so that when we train the model again or save it or debug it it will be easier.

vocabulary = list(unique_vocabulary)
vocabulary = ["<PAD>", "<UNK>"] + vocabulary
# These are fixed:
# <PAD> ---> 0
# <UNK> ---> 1

In [154]:
# ID mapping

# The goal is to create dictionary that maps each words into a unique ID.
# the method enumerate() gives you 2 things when used with a list: 
# 1- The index
# 2- The actual value (word) 

# (index, word)

# (0, "i")
# (1, "love")
# (2, "ai")
# (3, "python")

word_to_id = {}

for index, word in enumerate(vocabulary):
    word_to_id[word] = index




In [ ]:
# Numericalization

# In this step, we want to convert all sentences into numbers according to the ID map dictionary.
# Before we had only a word --> ID, we want the whole sentence to be a numbers not just one word. 

def encode_sentence(sentence, word_to_id):
    tokens = sentence.split()
    # [the, oil]

    ids = []


    # It will take the word like: oil, then it will look for the coresponding ID and get it and then append it.
    # Will do this for all words in the sentence and then it will return the numbers array.
    for word in tokens:
        ids.append(word_to_id.get(word, word_to_id["<UNK>"])) 
        # If the word exits in the vocabulary (which is built by using the training dataset) get the ID of that
        # word else return the ID of unknown word which is 1 (fixed)

    return ids



# For all sentences in the training set

X_train_ids = [] # This list will now have: [ [5575, 2875, 2542] --> sentence 1, [654, 1246, 756] --> sentence 2, ... ]

X_val_ids = [] # Also the validation set need to be encoded with the same vocabulary, so the model do not get 
# get confused if we used other vocabulary.

X_test_ids = [] # Also the training set need to be encoded

for sentence in X_train:
    ids = encode_sentence(sentence, word_to_id)
    X_train_ids.append(ids)

for sentence in X_val:
    ids = encode_sentence(sentence, word_to_id)
    X_val_ids.append(ids)

for sentence in X_test:
    ids = encode_sentence(sentence, word_to_id)
    X_test_ids.append(ids)



In [157]:
print(encode_sentence("the apple oil", word_to_id))

[93249, 112488, 50744]


In [ ]:
# Padding sequences

# Every sentence has a different lenght. So we need to make them with same length for regtangle tensor.

# Manually part 



# max_length = 6

# sequences = [
#     [5, 10, 22],
#     [4, 8, 15, 9, 33, 50]
# ]

# padded_sequnces = []

# for sequence in sequences:

#     padding_length = max_length - len(sequence)

#     padded_seq = sequence + [0] * padding_length # [0] * 3 --> [0,0,0]

#     padded_sequnces.append(padded_seq)



max_length = 100

X_train_padded = [] # After padding they will be saved here 
X_test_padded = []
X_val_padded = []

PAD_ID = word_to_id["<PAD>"] # get the pad id from the vocabulary 

for sentence in X_train_ids:

    # If sentence is longer, cut it
    sentence = sentence[:max_length] # The colon : means "go from the very start (index 0) up to, but not including, the index max_length".

    # If sentence is shorter, add padding 

    padding_length = max_length - len(sentence)

    sentence = sentence + [PAD_ID] * padding_length

    X_train_padded.append(sentence)

for sentence in X_test_ids:

    # If sentence is longer, cut it
    sentence = sentence[:max_length] # The colon : means "go from the very start (index 0) up to, but not including, the index max_length".

    # If sentence is shorter, add padding 

    padding_length = max_length - len(sentence)

    sentence = sentence + [PAD_ID] * padding_length

    X_test_padded.append(sentence)


for sentence in X_val_ids:

    # If sentence is longer, cut it
    sentence = sentence[:max_length] # The colon : means "go from the very start (index 0) up to, but not including, the index max_length".

    # If sentence is shorter, add padding 

    padding_length = max_length - len(sentence)

    sentence = sentence + [PAD_ID] * padding_length

    X_val_padded.append(sentence)


# Why padding the input (x) and not also padding y (labels) ? because X = the sentence and LTSM needs it to be a tensor 
# but y the label is a single class, there is no variable length. 


In [166]:
X_train_padded = np.array(X_train_padded)
print(X_train_padded.shape)

print(len(X_train_padded[0]))
print(len(X_val_padded[1]))
print(len(X_test_padded[100]))

(120000, 100)
100
100
100


### Neural Network 

In [ ]:
# Convert integers to tensors


X_train_tensor = torch.tensor(X_train_padded, dtype=torch.long)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)

X_test_tensor = torch.tensor(X_test_padded, dtype=torch.long)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

X_vali_tensor = torch.tensor(X_val_padded, dtype=torch.long)
y_vali_tensor = torch.tensor(y_val, dtype=torch.long)

In [ ]:
class AGNewsDataset(Dataset):

    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, index):
        return self.X[index], self.y[index]

In [ ]:
training_data = AGNewsDataset(X_train_tensor, y_train_tensor)
validation_data = AGNewsDataset(X_val, y_val)
testing_data = AGNewsDataset(X_test, y_test)